In [1]:
!unzip -q /content/data_SS_vs_WB.zip -d /content/
!ls /content/data_SS_vs_WB

classes.txt  images  labels  notes.json


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.0 MB/s eta 0:00:00


In [3]:
import random
import shutil
from pathlib import Path

DATA_DIR = Path("/content/data_SS_vs_WB")

images_dir = DATA_DIR / "images"
labels_dir = DATA_DIR / "labels"

output_dir = DATA_DIR / "yolo_split"

# Delete old split if it exists
if output_dir.exists():
    shutil.rmtree(output_dir)

# Create folders
for split in ["train", "test"]:
    (output_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (output_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

image_exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

images = [
    p for p in images_dir.iterdir()
    if p.suffix.lower() in image_exts
]

random.seed(42)
random.shuffle(images)

train_size = int(len(images) * 0.8)

train_images = images[:train_size]
test_images = images[train_size:]

def copy_data(image_list, split):
    missing_labels = 0

    for img_path in image_list:
        label_path = labels_dir / f"{img_path.stem}.txt"

        shutil.copy2(img_path, output_dir / "images" / split / img_path.name)

        if label_path.exists():
            shutil.copy2(label_path, output_dir / "labels" / split / label_path.name)
        else:
            missing_labels += 1
            print(f"Missing label: {img_path.name}")

    return missing_labels

missing_train = copy_data(train_images, "train")
missing_test = copy_data(test_images, "test")

print("Split complete")
print(f"Total images: {len(images)}")
print(f"Train images: {len(train_images)}")
print(f"Test images: {len(test_images)}")
print(f"Missing train labels: {missing_train}")
print(f"Missing test labels: {missing_test}")

Split complete
Total images: 194
Train images: 155
Test images: 39
Missing train labels: 0
Missing test labels: 0


In [4]:
from pathlib import Path
from collections import Counter

label_dir = Path("/content/data_SS_vs_WB/labels")
counts = Counter()

for txt in label_dir.glob("*.txt"):
    text = txt.read_text().strip()
    if not text:
        continue

    for line in text.splitlines():
        cls_id = int(float(line.split()[0]))
        counts[cls_id] += 1

print("Class ID counts:", counts)

Class ID counts: Counter({1: 851, 2: 815, 0: 209})


In [ ]:
yaml_text = """
path: /content/data_SS_vs_WB/yolo_split
train: images/train
val: images/test
test: images/test

names:
  0: REF
  1: SS
  2: WB
"""

with open("/content/data.yaml", "w") as f: ac
    f.write(yaml_text)

print(open("/content/data.yaml").read())


path: /content/data_SS_vs_WB/yolo_split
train: images/train
val: images/test
test: images/test

names:
  0: REF
  1: SS
  2: WB



In [6]:
!echo "Train images:"
!find /content/data_SS_vs_WB/yolo_split/images/train -type f | wc -l

!echo "Test images:"
!find /content/data_SS_vs_WB/yolo_split/images/test -type f | wc -l

!echo "Train labels:"
!find /content/data_SS_vs_WB/yolo_split/labels/train -type f | wc -l

!echo "Test labels:"
!find /content/data_SS_vs_WB/yolo_split/labels/test -type f | wc -l

Train images:
155
Test images:
39
Train labels:
155
Test labels:
39


In [7]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.49 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7db0881082c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [11]:
from google.colab import files

files.download("/content/runs/detect/train/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
!pip install ultralytics opencv-python -q

In [13]:
model_path = "/content/runs/detect/train/weights/best.pt"

In [14]:
from ultralytics import YOLO

model_path = "/content/runs/detect/train/weights/best.pt"
model = YOLO(model_path)

print("Model loaded successfully")

Model loaded successfully


In [15]:
from pathlib import Path

model_path = Path("/content/runs/detect/train/weights/best.pt")
video_path = Path("/content/sample.mp4")

print("Model exists:", model_path.exists(), model_path)
print("Video exists:", video_path.exists(), video_path)

Model exists: True /content/runs/detect/train/weights/best.pt
Video exists: True /content/sample.mp4


In [19]:
from ultralytics import YOLO
import cv2
from pathlib import Path

# Paths
model_path = "/content/runs/detect/train/weights/best.pt"
video_path = "/content/sample.mp4"
output_path = "/content/sample_tracked_custom.mp4"

# Load model
model = YOLO(model_path)

# Class colors in BGR format for OpenCV
# REF = green, WB = blue, SS = pink
CLASS_COLORS = {
    "REF": (0, 255, 0),
    "WB": (255, 0, 0),
    "SS": (255, 0, 255),
}

def get_class_color(class_name):
    return CLASS_COLORS.get(class_name, (255, 255, 255))

def draw_counts(frame, counts):
    text = f"REF: {counts['REF']} | WB: {counts['WB']} | SS: {counts['SS']}"

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.8
    thickness = 2

    text_size, _ = cv2.getTextSize(text, font, font_scale, thickness)
    text_width, text_height = text_size

    x = 20
    y = 40

    # Background box
    cv2.rectangle(
        frame,
        (x - 10, y - text_height - 10),
        (x + text_width + 10, y + 10),
        (0, 0, 0),
        -1
    )

    cv2.putText(
        frame,
        text,
        (x, y),
        font,
        font_scale,
        (255, 255, 255),
        thickness
    )

    return frame

def draw_tracked_boxes(frame, result):
    counts = {
        "REF": 0,
        "WB": 0,
        "SS": 0,
    }

    if result.boxes is None or len(result.boxes) == 0:
        return draw_counts(frame, counts)

    names = result.names

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])

        class_name = names[cls_id]

        if class_name not in counts:
            continue

        counts[class_name] += 1
        color = get_class_color(class_name)

        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        track_id = None
        if box.id is not None:
            track_id = int(box.id[0])

        if track_id is not None:
            label = f"{class_name} ID:{track_id} {conf:.2f}"
        else:
            label = f"{class_name} {conf:.2f}"

        # Draw bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

        # Draw label background
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.6
        thickness = 2

        label_size, _ = cv2.getTextSize(label, font, font_scale, thickness)
        label_width, label_height = label_size

        label_y1 = max(y1 - label_height - 10, 0)
        label_y2 = max(y1, label_height + 10)

        cv2.rectangle(
            frame,
            (x1, label_y1),
            (x1 + label_width + 10, label_y2),
            color,
            -1
        )

        cv2.putText(
            frame,
            label,
            (x1 + 5, label_y2 - 5),
            font,
            font_scale,
            (255, 255, 255),
            thickness
        )

    return draw_counts(frame, counts)

# Open video
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Could not open video. Make sure sample.mp4 is at /content/sample.mp4")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS:", fps)
print("Width:", width)
print("Height:", height)
print("Total frames:", total_frames)

# Save output video
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

frame_num = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    results = model.track(
        source=frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.25,
        iou=0.5,
        verbose=False
    )

    annotated_frame = draw_tracked_boxes(frame, results[0])

    writer.write(annotated_frame)

    frame_num += 1

    if frame_num % 50 == 0:
        print(f"Processed {frame_num}/{total_frames} frames")

cap.release()
writer.release()

print("Done.")
print("Saved video to:", output_path)

FPS: 24.95438219080729
Width: 1280
Height: 720
Total frames: 406
Processed 50/406 frames
Processed 100/406 frames
Processed 150/406 frames
Processed 200/406 frames
Processed 250/406 frames
Processed 300/406 frames
Processed 350/406 frames
Done.
Saved video to: /content/sample_tracked_custom.mp4


In [20]:
from google.colab import files

files.download("/content/sample_tracked_custom.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>